# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zeeofficial01/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Freshness and page performance

The FlyRank research report observed that, among pages older than 365 days, pages refreshed in the last 30 days had higher measured health and impressions than pages last updated 181–360 days ago. The report describes this as an observed pattern rather than proof of cause and effect.

**My methodology question:** Where does the “refreshed” label come from, and could pages selected for refreshing already differ from stale pages in ways that affect impressions or health? For example, pages chosen for refresh may have had different levels of existing visibility, importance, or attention. I would want to understand how the comparison accounts for these differences before treating the measured lift as evidence that refreshing itself caused the change.

### Finding 2 — AI-generated content and performance

The FlyRank report observed that its dataset was largely AI-generated and did not show a blanket performance penalty associated only with AI use. It compared AI model cohorts within age groups and found that different models performed differently across cohorts.

**My methodology question:** How is the “AI-generated” label determined, and does the comparison sufficiently account for other factors such as publication age, topic, and editing or publishing quality? I would want to know how consistently the labels were assigned and how much these other factors could contribute to the observed differences before extending the finding beyond this dataset.

### Scope of my questions

These are methodology questions rather than judgments about whether the findings are correct. The goal is to identify what information about labels, comparison groups, and validation design would help determine how far the evidence can support each claim.


## 2. My model under an honest split (before/after)

### What Week 5 actually established

The Week-5 notebook did not produce a supervised classification score because the March 2026 dataset does not contain the original `is_declining_label` target. I did not infer a replacement label from the same features because that would change the prediction task and could create a misleading evaluation.

The Week-5 analysis therefore produced a rule-based baseline signal using CTR relative to position-bucket reference CTR and an impressions-volume threshold. The analysis used 176,738 content items with complete modeling signals.

### Honest split design

For this audit, I use `client_hash_id` as the grouping variable. Content from the same client should not appear in both the training and evaluation groups when testing whether a signal generalizes across clients.

Because there is no valid supervised target in the Week-5 dataset, I do not report an invented model F1 score. Instead, I compare the construction of the baseline signal under the original row-level analysis and under a client-grouped design.

This makes the limitation explicit: the result is a validation of the evaluation design and signal construction, not evidence of supervised predictive performance.

### Before / after

**Before:** Week-5 analyzed content-level rows together and calculated position-bucket reference CTR and the impressions threshold from the full modeling dataset.

**After:** This audit separates clients into non-overlapping groups before calculating the reference statistics for the training portion and applies those learned reference values to held-out clients.

The key question is whether the observed signal remains similar when information from the evaluation clients is not used to construct the reference statistics.

### Interpretation

The grouped design is a more conservative test of cross-client generalization. Any difference between the original and grouped results is treated as an observed measurement of sensitivity to the validation design, not as proof that the model would perform at a particular level in production.


In [2]:
# ML-09 — Recreate the Week-5 modeling dataset

import numpy as np
import pandas as pd
from datasets import load_dataset

# Load the same March 2026 dataset used in Week 5
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(dataset)

# Convert to pandas
df_raw = dataset["train"].to_pandas()

# Aggregate daily performance to content level
df = (
    df_raw
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
    )
)

# Calculate CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

# Same modeling features used in Week 5
model_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ctr"
]

# Keep only rows with complete modeling signals
df_model = df.dropna(subset=model_features).copy()

print("Content-level rows:", len(df))
print("Rows available for modeling:", len(df_model))
print("Rows removed:", len(df) - len(df_model))

print("\nModeling columns:")
print(df_model[model_features].head())

# Confirm the original target is unavailable
print(
    "\nis_declining_label available:",
    "is_declining_label" in df_model.columns
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 9841378
    })
})
Content-level rows: 331437
Rows available for modeling: 176738
Rows removed: 154699

Modeling columns:
    gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions       ctr
3                 1           0          9.000000           0.0  0.000000
7               331           2         14.129210           0.0  0.006042
8                33           0          9.225529          

In [3]:
# ML-09 — Honest grouped-by-client comparison
# No supervised target is available in the Week-5 dataset,
# so this compares the rule-based signal construction rather
# than inventing a predictive F1 score.

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd

# Use the Week-5 modeling dataset
audit_df = df_model.copy()

# Keep client identity only for grouping; do not use it as a model feature.
groups = audit_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(audit_df, groups=groups)
)

train_df = audit_df.iloc[train_idx].copy()
test_df = audit_df.iloc[test_idx].copy()

print("Total rows:", len(audit_df))
print("Training rows:", len(train_df))
print("Held-out rows:", len(test_df))
print("Training clients:", train_df["client_hash_id"].nunique())
print("Held-out clients:", test_df["client_hash_id"].nunique())
print(
    "Client overlap:",
    len(
        set(train_df["client_hash_id"])
        & set(test_df["client_hash_id"])
    )
)

Total rows: 176738
Training rows: 138310
Held-out rows: 38428
Training clients: 37
Held-out clients: 10
Client overlap: 0


In [4]:
# ML-09 — Before/after comparison of baseline reference construction

# -----------------------------
# BEFORE: Week-5 construction
# -----------------------------

before_df = audit_df.copy()

before_df["position_bucket"] = pd.cut(
    before_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

before_reference = (
    before_df
    .groupby("position_bucket", observed=True)
    .agg(reference_ctr=("ctr", "mean"))
    .reset_index()
)

before_df = before_df.merge(
    before_reference,
    on="position_bucket",
    how="left"
)

before_df["ctr_gap"] = (
    before_df["reference_ctr"] - before_df["ctr"]
)

before_df["ctr_signal"] = (
    before_df["ctr_gap"] > 0
).astype(int)

before_volume_threshold = (
    before_df["gsc_impressions"].quantile(0.75)
)

before_df["volume_signal"] = (
    before_df["gsc_impressions"] >= before_volume_threshold
).astype(int)

before_df["baseline_score"] = (
    before_df["ctr_signal"] +
    before_df["volume_signal"]
)


# -----------------------------
# AFTER: grouped construction
# -----------------------------

train_audit = train_df.copy()
test_audit = test_df.copy()

train_audit["position_bucket"] = pd.cut(
    train_audit["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

test_audit["position_bucket"] = pd.cut(
    test_audit["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# Learn reference CTR only from training clients
after_reference = (
    train_audit
    .groupby("position_bucket", observed=True)
    .agg(reference_ctr=("ctr", "mean"))
    .reset_index()
)

# Apply training reference values to held-out clients
test_audit = test_audit.merge(
    after_reference,
    on="position_bucket",
    how="left"
)

test_audit["ctr_gap"] = (
    test_audit["reference_ctr"] - test_audit["ctr"]
)

test_audit["ctr_signal"] = (
    test_audit["ctr_gap"] > 0
).astype(int)

# Learn volume threshold only from training clients
after_volume_threshold = (
    train_audit["gsc_impressions"].quantile(0.75)
)

test_audit["volume_signal"] = (
    test_audit["gsc_impressions"] >= after_volume_threshold
).astype(int)

test_audit["baseline_score"] = (
    test_audit["ctr_signal"] +
    test_audit["volume_signal"]
)


# -----------------------------
# Comparison
# -----------------------------

print("BEFORE — Full dataset reference construction")
print("Position reference CTR:")
print(before_reference)
print("Volume threshold:", before_volume_threshold)

print("\nAFTER — Training-only reference construction")
print("Position reference CTR:")
print(after_reference)
print("Volume threshold:", after_volume_threshold)

print("\nHELD-OUT CLIENTS — Signal distribution")
print(
    test_audit["baseline_score"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("share")
)

print("\nHeld-out rows:", len(test_audit))
print(
    "Held-out signal-2 rows:",
    (test_audit["baseline_score"] == 2).sum()
)

BEFORE — Full dataset reference construction
Position reference CTR:
  position_bucket  reference_ctr
0             1-3       0.012399
1            4-10       0.004926
2           11-20       0.003211
3             21+       0.001928
Volume threshold: 1039.0

AFTER — Training-only reference construction
Position reference CTR:
  position_bucket  reference_ctr
0             1-3       0.014546
1            4-10       0.005235
2           11-20       0.003292
3             21+       0.001923
Volume threshold: 912.0

HELD-OUT CLIENTS — Signal distribution
baseline_score
0    0.082388
1    0.624336
2    0.293276
Name: share, dtype: float64

Held-out rows: 38428
Held-out signal-2 rows: 11270


### Before/after interpretation

The grouped split produced 38,428 held-out content items from 10 clients, with zero client overlap between training and held-out groups.

Under the original Week-5 construction, the position-bucket reference CTRs were calculated from the full modeling dataset. The 75th-percentile impressions threshold was 1,039.

Under the grouped construction, the reference CTRs and impressions threshold were calculated using training clients only, and then applied to the held-out clients. The training-only impressions threshold was 912.

The position-bucket reference CTRs also changed. For example, the 1–3 position bucket changed from 0.012399 to 0.014546, while the 4–10 bucket changed from 0.004926 to 0.005235.

For the held-out clients, 8.24% of items received baseline score 0, 62.43% received score 1, and 29.33% received score 2. There were 11,270 held-out items with score 2.

These differences show that the baseline signal is sensitive to how its reference statistics are constructed. The grouped design is therefore a more conservative validation setup because the held-out clients do not contribute to the reference values used to score them.

This is an observed sensitivity check, not a supervised model performance result. Because the original decline label is unavailable, these measurements should be treated as directional decision-support evidence rather than predictive accuracy.


## 3. Leakage audit

I reviewed the final Week-5 feature and signal construction for possible leakage.

The main question is whether a feature uses information that would only become available after the decision or review point, or whether information from the evaluation clients is used when constructing the reference statistics.

| Feature / signal   | Leakage risk | Audit finding                                                                                 | Action                                                           |
| ------------------ | ------------ | --------------------------------------------------------------------------------------------- | ---------------------------------------------------------------- |
| `gsc_impressions`  | Low          | A measured performance signal from the available March dataset.                               | Retained                                                         |
| `gsc_clicks`       | Low          | A measured performance signal from the available March dataset.                               | Retained                                                         |
| `gsc_avg_position` | Low          | A measured search-position signal.                                                            | Retained                                                         |
| `ga4_sessions`     | Low          | A measured traffic signal.                                                                    | Retained                                                         |
| `ctr`              | Low          | Calculated from GSC clicks and impressions from the same observation period.                  | Retained                                                         |
| `reference_ctr`    | Medium       | Can become leakage if calculated using held-out clients.                                      | Recomputed from training clients only in the honest split        |
| `ctr_gap`          | Medium       | Depends on `reference_ctr`, so it inherits its leakage risk.                                  | Calculated using the training-only reference in the honest split |
| `ctr_signal`       | Medium       | Depends on `ctr_gap`.                                                                         | Calculated after the grouped reference construction              |
| `volume_signal`    | Medium       | The 75th-percentile threshold can use held-out information if calculated on the full dataset. | Threshold learned from training clients only in the honest split |
| `baseline_score`   | Medium       | Combines signals that depend on the reference statistics and volume threshold.                | Recomputed after the grouped construction                        |

### Main leakage finding

The main issue identified was not a single raw feature, but the way aggregate reference values were constructed.

In the original Week-5 analysis, the position-bucket reference CTR and volume threshold were calculated from the complete modeling dataset. That means information from clients later treated as held out could contribute to the reference values.

For the honest grouped analysis, these values are learned using the training clients only and then applied to the held-out clients. This prevents the held-out clients from influencing the reference statistics used to score them.

I did not identify evidence that the raw GSC or GA4 measurements themselves contain a direct target label. The main methodological risk is therefore reference-statistic leakage rather than a reconstructed decline label.

### Scope

This audit checks leakage in the features and signal construction available in the Week-5 March dataset. It does not prove that the data is free of every possible temporal or measurement issue. The absence of the original decline label also means this notebook does not establish supervised predictive performance.


In [5]:
# ML-09 — Leakage audit: verify grouped construction

print("RAW / DIRECT FEATURES")
raw_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ctr"
]

for feature in raw_features:
    print(f"{feature}: available = {feature in audit_df.columns}")

print("\nGROUPING CHECK")
train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print("Training clients:", len(train_clients))
print("Held-out clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

print("\nREFERENCE CONSTRUCTION")
print("Before threshold:", before_volume_threshold)
print("After threshold:", after_volume_threshold)

print("\nREFERENCE CTR — BEFORE")
display(before_reference)

print("\nREFERENCE CTR — AFTER")
display(after_reference)

print("\nLEAKAGE CHECK RESULT")

if len(train_clients & test_clients) == 0:
    print("PASS: No client appears in both training and held-out groups.")
else:
    print("CHECK REQUIRED: Client overlap detected.")

print(
    "PASS: Held-out scoring uses reference CTR learned from training clients only."
)

print(
    "PASS: Held-out scoring uses the volume threshold learned from training clients only."
)

RAW / DIRECT FEATURES
gsc_impressions: available = True
gsc_clicks: available = True
gsc_avg_position: available = True
ga4_sessions: available = True
ctr: available = True

GROUPING CHECK
Training clients: 37
Held-out clients: 10
Client overlap: 0

REFERENCE CONSTRUCTION
Before threshold: 1039.0
After threshold: 912.0

REFERENCE CTR — BEFORE


,position_bucket,reference_ctr
0,1-3,0.012399
1,4-10,0.004926
2,11-20,0.003211
3,21+,0.001928



REFERENCE CTR — AFTER


,position_bucket,reference_ctr
0,1-3,0.014546
1,4-10,0.005235
2,11-20,0.003292
3,21+,0.001923



LEAKAGE CHECK RESULT
PASS: No client appears in both training and held-out groups.
PASS: Held-out scoring uses reference CTR learned from training clients only.
PASS: Held-out scoring uses the volume threshold learned from training clients only.


## 4. Claim rewrite

The Week-5 analysis should be described as an observed, directional signal rather than as proof of predictive performance or causation.

### Claim 1 — Baseline score

**Overly strong version:**
The baseline score identifies declining content and predicts which pages will perform poorly.

**Safer version:**
The baseline score provides a directional signal for prioritizing content based on observed CTR relative to position-bucket reference values and search-impression volume. It should be treated as decision-support evidence rather than a validated predictor of future decline because the original decline label is unavailable.

### Claim 2 — CTR reference comparison

**Overly strong version:**
Pages with CTR below the reference CTR are underperforming and should be fixed.

**Safer version:**
Pages with CTR below the position-bucket reference CTR can be flagged for further review. This is an observed comparison within the available dataset and does not by itself establish that the page is underperforming for reasons caused by the content or that a specific intervention will improve performance.

### Claim 3 — Validation result

**Overly strong version:**
The model is accurate and generalizes to new clients.

**Safer version:**
The grouped-by-client analysis provides a more conservative validation setup by keeping held-out clients separate from the clients used to construct the reference statistics. Because no ground-truth decline label is available, this analysis does not establish predictive accuracy or generalization performance.

### Claim 4 — Overall decision use

**Overly strong version:**
The analysis proves which content should be changed.

**Safer version:**
The analysis can support content-prioritization decisions by highlighting items with combinations of lower-than-reference CTR and higher impression volume. Final decisions should incorporate additional evidence and human review.

### Language rule used

Throughout this notebook, I use terms such as **observed**, **measured**, **directional**, **signal**, and **decision-support**. I avoid describing the baseline as a validated predictive model because the target label required for supervised evaluation is unavailable.


## 5. Self-check

* [x] Two research findings were identified and a constructive methodology question was written for each.
* [x] The Week-5 analysis was recreated using the available March 2026 dataset.
* [x] An honest grouped-by-client split was performed.
* [x] Training and held-out clients had zero overlap.
* [x] Reference CTR values were learned from training clients only for held-out scoring.
* [x] The impressions threshold was learned from training clients only.
* [x] Feature and signal construction was reviewed for leakage.
* [x] No missing `is_declining_label` target was invented or reconstructed.
* [x] Claims were rewritten using cautious language such as observed, measured, directional, and decision-support.
* [x] No client names, private queries, or private URLs are included.
* [x] The notebook is intended to run from top to bottom in a fresh runtime.
* [x] The notebook belongs under `work/notebooks/w06_validation_audit.ipynb`.

### Final conclusion

The Week-5 baseline provides a useful directional content-prioritization signal, but the available dataset does not contain the original decline label needed for supervised performance evaluation. The grouped-by-client validation reduces contamination from client-level reference statistics and provides a more conservative assessment of the signal construction.

The audit found that the original full-dataset reference construction could allow held-out information to influence the reference values. The corrected grouped approach learns those values from training clients only before applying them to held-out clients.

Therefore, the appropriate interpretation is that the analysis provides **observed and measured decision-support evidence**, not proof of causality or validated predictive accuracy.
